# Validations and Cleaning

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

from src.data_loading import YEARS, load_year
from src import validations as V
from src import cleaning as C

frames = {y: load_year(y) for y in YEARS}
{y: f.shape for y, f in frames.items()}

{2024: (65437, 115), 2025: (49191, 173)}

# Validation of the Raw Data

## A1. Duplicate Answers

In [3]:
V.check_duplicates(frames)

,rows,duplicate_ResponseId
2024,65437,0
2025,49191,0


## A2. WorkExp

In [4]:
summary, coherence = V.check_experience(frames)
display(summary)
display(coherence)

,dtype_original,pct_missing,min,median,max
2024,float64,54.7,0.0,9.0,50.0
2025,float64,12.8,1.0,10.0,100.0


,valid_pairs,correlation,diferenca_absoluta_median,WorkExp_missing_but_YearsCodePro_present
2024,28852.0,0.928,0.0,22758.0


## A3. Emplouyment Values

In [5]:
counts, kept = V.check_employment(frames)
for y, t in counts.items():
    print(y); display(t)
display(kept)

2024


,n
Employment,
"Employed, full-time",39041
"Independent contractor, freelancer, or self-employed",4846
"Student, full-time",4709
"Employed, full-time;Independent contractor, freelancer, or self-employed",3557
"Not employed, but looking for work",2341
"Employed, part-time",1266
"Student, full-time;Employed, part-time",1115
"Employed, full-time;Student, full-time",897
"Employed, full-time;Student, part-time",839


2025


,n
Employment,
Employed,33750
"Independent contractor, freelancer, or self-employed",6708
Student,4428
Not employed,2227
NaN,852
Retired,708
I prefer not to say,518


,rows,retained_by_filter,pct_retained
2024,65437.0,48801.0,74.6
2025,49191.0,33750.0,68.6


## A4. Who answered AIAcc, AISent, and AIComplex?

In [6]:
base = V.check_ai_base(frames)
for(y,col), t in base.items():
    print(y, col); display(t)

2024 AIAcc


,no_response,responded,pct_responded
ai_use_group,,,
(missing),4530,0,0.0
no,14837,0,0.0
plans,8408,0,0.0
uses,360,37302,99.0


2024 AISent


,no_response,responded,pct_responded
ai_use_group,,,
(missing),4530,0,0.0
no,14837,0,0.0
plans,71,8337,99.2
uses,126,37536,99.7


2024 AIComplex


...

,no_response,responded,pct_responded
ai_use_group,,,
(missing),15434,37,0.2
no,74,5380,98.6
plans,43,1754,97.6
uses,343,26126,98.7


2025 AISent


,no_response,responded,pct_responded
ai_use_group,,,
(missing),15428,43,0.3
no,29,5425,99.5
plans,28,1769,98.4
uses,239,26230,99.1


2025 AIComplex


,no_response,responded,pct_responded
ai_use_group,,,
(missing),15432,39,0.3
no,77,5377,98.6
plans,58,1739,96.8
uses,341,26128,98.7


## A5. Missing values in AI columns

In [7]:
V.check_missing_ai(frames)

,2024,2025
AISelect,6.9,31.5
AISent,29.9,32.0
AIAcc,43.0,32.3
AIComplex,43.4,32.3
AIThreat,31.7,26.7


## A6. Countries that appear in only one year (naming)

In [8]:
V.check_countries(frames)

,2024,2025
Country,,
Moldova,0,9
Saint Lucia,0,2
Timor-Leste,0,1
Gambia,0,1
Palau,0,1
Bahamas,4,0
Burkina Faso,4,0
Equatorial Guinea,1,0
Central African Republic,1,0


## A7. Compensation (ConvertedCompYearly)

In [9]:
V.check_compensation(frames)

,pct_missing,q0,q1,q25,q50,q75,q99,q100
2024,64.2,1.0,208.0,32712.0,65000.0,107972.0,393751.0,16256603.0
2025,51.3,1.0,65.0,38171.0,75320.0,120596.0,440856.0,50000000.0


## Run the cleaning pipeline

In [10]:
df,log = C.build_clean_dataset(frames=frames)
display(log)
print(df.shape)

,step,total,2024,2025
0,0. raw (mapped columns),114628,65437,49191
1,1. sem ResponseId duplicado,114628,65437,49191
2,2. empregados (Employed),82551,48801,33750
3,3. valid experience (WorkExp),75936,43744,32192
4,4. known country,67141,42628,24513
5,"5. valid compensation, no outliers",39182,20669,18513


(39182, 38)


## D. Validate the results

### D1. Compensation by year

In [11]:
df.groupby("year")["comp_yearly"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
year,,,,,,,,
2024,20669.0,79989.0,60985.0,1728.0,36798.0,66592.0,107406.0,382164.0
2025,18513.0,94301.0,68148.0,2092.0,46986.0,81210.0,125000.0,449267.0


### D2. AI adoption by year (preserving NaN)


In [12]:
df.groupby("year")["uses_ai"].agg(adoption_rate="mean", n_responses ="count", n_missing=lambda s: s.isna().sum())

,adoption_rate,n_responses,n_missing
year,,,
2024,0.626445,20669,0
2025,0.807839,18037,476


### D3: Distribution by use group and frequency (2025)

In [13]:
display(pd.crosstab(df["year"], df["ai_use_group"].fillna("(missing)"), normalize="index").round(3))
display(df.loc[df["year"] == 2025, "ai_use_frequency"].value_counts(dropna=False))

ai_use_group,(missing),no,plans,uses
year,,,,
2024,0.000,0.243,0.131,0.626
2025,0.026,0.143,0.044,0.787


ai_use_frequency
daily              8721
<NA>               3942
weekly             3386
monthly_or_less    2464
Name: count, dtype: Int64

### D4. Ordinal scales: AI users only

In [14]:
users = df[df["uses_ai"] == True]
users.groupby("year")[["ai_sentiment_score", "ai_accuracy_score", "ai_complexity_score"]].agg(["mean", "count"]).round(2)

ai_sentiment_score        ai_accuracy_score        ai_complexity_score       
                   mean  count              mean  count                mean  count
year                                                                              
2024               3.97  12752              3.00  12934                2.74  12876
2025               3.89  14382              2.87  14497                2.72  13742

In [16]:
back = pd.read_parquet(C.OUTPUT)
print(C.OUTPUT, back.shape)
back.dtypes

data\processed\survey_clean.parquet (39182, 38)


response_id                     int64
employment                     object
country                        object
dev_type                       object
ed_level                       object
org_size                       object
remote_work                    object
years_pro                     float64
years_pro_alt                  object
years_code                    float64
comp_total                    float64
comp_yearly                   float64
job_sat                       float64
languages                      object
databases                      object
platforms                      object
webframes                      object
ai_use                         object
ai_sentiment                   object
ai_accuracy                    object
ai_complexity                  object
ai_threat                      object
ai_benefits                    object
ai_ethics                      object
ai_challenges                  object
year                            int64
ai_agents   